# 2장 — Attention에서 Transformer와 GPT까지

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch02_transformer_gpt.ipynb)

『밑바닥부터 시작하는 딥러닝 6』 공식 저장소의 `c9b6e2ed531b08dd9f451a091a34e9645148e2e2` 커밋을 기준으로 구성합니다. 알고리즘과 모델 구조는 원본을 유지하고, 논리적으로 함께 읽어야 하는 코드만 같은 셀에 묶습니다.

## 노트북 구성 원칙

1. 원본 `.py`의 계산 순서와 모델 구조를 유지합니다.
2. 변수 한 줄마다 셀을 나누지 않고 **설정 → 계산 → 결과 확인** 단위로 묶습니다.
3. 클래스는 역할이 분명하도록 클래스 단위로 나눕니다.
4. 주석은 한국어로 적되 변수명·수식·텐서 shape은 원본을 유지합니다.
5. 실행 결과는 실행된 셀 바로 아래에 남깁니다.

## 0. 실행 환경 확인

In [1]:
import sys
import torch

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA 사용 가능:', torch.cuda.is_available())
print('실행 장치:', 'cuda' if torch.cuda.is_available() else 'cpu')

Python: 3.13.5
PyTorch: 2.10.0+cpu
CUDA 사용 가능: False
실행 장치: cpu


## 1. 공통 구현 — `codebot/model.py`

뒤의 예제가 사용하는 공통 모델 구현입니다. 외부 파일 뒤에 숨기지 않고 클래스 단위로 직접 확인합니다.

### Multi-Head Attention

In [2]:
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_head, head_dim, dropout_rate=0.1):
        super().__init__()
        self.n_head = n_head
        self.head_dim = head_dim
        E, H, D = embed_dim, n_head, head_dim

        self.W_q = nn.Linear(E, H * D, bias=False)
        self.W_k = nn.Linear(E, H * D, bias=False)
        self.W_v = nn.Linear(E, H * D, bias=False)
        self.W_o = nn.Linear(H * D, E, bias=False)
        self.attention_dropout = nn.Dropout(dropout_rate)
        self.output_dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        B, C, E = x.shape
        H, D = self.n_head, self.head_dim

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        Q = Q.view(B, C, H, D).transpose(1, 2)
        K = K.view(B, C, H, D).transpose(1, 2)
        V = V.view(B, C, H, D).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (D ** 0.5)
        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        weights = self.attention_dropout(weights)
        hidden = torch.matmul(weights, V)
        hidden = hidden.transpose(1, 2).contiguous()
        hidden = hidden.view(B, C, H * D)

        output = self.W_o(hidden)
        return self.output_dropout(output)

### LayerNorm, GELU, FFN, Transformer Block

In [3]:
class LayerNorm(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(embed_dim))
        self.beta = nn.Parameter(torch.zeros(embed_dim))
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * norm_x + self.beta


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FFN(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout_rate):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout_rate),
        )

    def forward(self, x):
        return self.layers(x)


class Block(nn.Module):
    def __init__(self, embed_dim, n_head, ff_dim, dropout_rate=0.1):
        super().__init__()
        head_dim = embed_dim // n_head
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_head, head_dim, dropout_rate)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = FFN(embed_dim, ff_dim, dropout_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

## 2. 장별 실습 코드

## `ch02/01_soft_dict.py`

In [4]:
import numpy as np

d = {
    'apple': 100,
    'banana': 200,
    'cherry': 300,
    'durian': 400,
}
query = 'banana'
print(d[query])

movie_preferences = {
    (8, 2, 3): 85,
    (3, 9, 1): 70,
    (1, 2, 9): 60,
    (5, 5, 5): 75,
    (7, 6, 2): 80,
    (2, 7, 6): 65,
    (9, 1, 1): 90,
}
new_movie = (6, 4, 5)

def soft_dictionary(query, dictionary):
    similarity = []
    for key in dictionary:
        similarity.append(np.dot(query, key))

    exp_similarity = np.exp(similarity)
    weights = exp_similarity / np.sum(exp_similarity)

    result = 0
    for weight, value in zip(weights, dictionary.values()):
        result += weight * value
    return result, weights

predicted_rating, weights = soft_dictionary(new_movie, movie_preferences)
print(f'새 영화 {new_movie}의 예측 평가: {predicted_rating:.2f}점')
print('각 영화의 가중치:')
for key, weight in zip(movie_preferences.keys(), weights):
    print(f'영화 {key}: {weight * 100:.2f}%')

200
새 영화 (6, 4, 5)의 예측 평가: 78.66점
각 영화의 가중치:
영화 (8, 2, 3): 0.49%
영화 (3, 9, 1): 0.00%
영화 (1, 2, 9): 0.00%
영화 (5, 5, 5): 26.71%
영화 (7, 6, 2): 72.62%
영화 (2, 7, 6): 0.18%
영화 (9, 1, 1): 0.00%


## `ch02/02_attn_math.py`

In [5]:
K = torch.tensor([
    [8, 2, 3], [3, 9, 1], [1, 2, 9], [5, 5, 5],
    [7, 6, 2], [2, 7, 6], [9, 1, 1],
], dtype=torch.float32)
V = torch.tensor([[85], [70], [60], [75], [80], [65], [90]], dtype=torch.float32)
Q = torch.tensor([[6, 4, 5], [2, 8, 3], [4, 3, 7]], dtype=torch.float32)

def attention(Q, K, V):
    similarity = torch.matmul(Q, K.t())
    weights = F.softmax(similarity, dim=1)
    output = torch.matmul(weights, V)
    return output, weights

predicted_ratings, weights = attention(Q, K, V)
for movie, rating in zip(Q, predicted_ratings):
    print(f'영화 {movie.numpy()}의 예측 평가: {rating.item():.2f}')

영화 [6. 4. 5.]의 예측 평가: 78.66
영화 [2. 8. 3.]의 예측 평가: 69.76
영화 [4. 3. 7.]의 예측 평가: 61.20


## `ch02/03_attn_scaling.py`

In [6]:
x = torch.tensor([100.0, 200.0, 300.0])
y = F.softmax(x, dim=0)
print(y)

tensor([0.0000e+00, 3.7835e-44, 1.0000e+00])


In [7]:
import matplotlib.pyplot as plt

d = 10
q = np.random.randn(d)
k = np.random.randn(d)
dot_product = np.dot(q, k)
scaled_dot_product = dot_product / np.sqrt(d)

print('dot product:', dot_product)
print('scaled dot product:', scaled_dot_product)

dot product: 3.2864620945653606
scaled dot product: 1.0392705662634218


In [8]:
num_samples = 10000
dot_products = []
scaled_dot_products = []

for _ in range(num_samples):
    q = np.random.randn(d)
    k = np.random.randn(d)
    dot_product = np.dot(q, k)
    scaled_dot_product = dot_product / np.sqrt(d)
    dot_products.append(dot_product)
    scaled_dot_products.append(scaled_dot_product)

plt.figure(figsize=(10, 6))
plt.hist(dot_products, bins=50, alpha=0.5, label='Without scaling')
plt.hist(scaled_dot_products, bins=50, alpha=0.5, label='With scaling')
plt.legend()
plt.show()

print('Variances without scaling:', np.var(dot_products))
print('Variances with scaling:', np.var(scaled_dot_products))

Variances without scaling: 10.055257939500734
Variances with scaling: 1.0055257939500735


## `ch02/06_attn_mask.py`

In [9]:
class Attention(nn.Module):
    def __init__(self, embed_dim, key_dim):
        super().__init__()
        self.W_q = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.key_dim = key_dim

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (self.key_dim ** 0.5)

        B, C, E = x.shape
        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        return torch.matmul(weights, V)

attention = Attention(embed_dim=256, key_dim=64)
x = torch.randn(2, 5, 256)
y = attention(x)
print('입력 shape:', x.shape)
print('출력 shape:', y.shape)

입력 shape: torch.Size([2, 5, 256])
출력 shape: torch.Size([2, 5, 256])


## `ch02/07_attn_value.py`

In [10]:
class Attention(nn.Module):
    def __init__(self, embed_dim, key_dim):
        super().__init__()
        self.W_q = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, key_dim, bias=False)
        self.W_o = nn.Linear(key_dim, embed_dim, bias=False)
        self.key_dim = key_dim

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (self.key_dim ** 0.5)

        B, C, E = x.shape
        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        hidden = torch.matmul(weights, V)
        return self.W_o(hidden)

attention = Attention(embed_dim=256, key_dim=64)
x = torch.randn(2, 5, 256)
y = attention(x)
print('입력 shape:', x.shape)
print('출력 shape:', y.shape)

입력 shape: torch.Size([2, 5, 256])
출력 shape: torch.Size([2, 5, 256])


## `ch02/08_multi_head.py`

In [11]:
B = 2
C = 4
E = 16
H = 3
D = 8

x = torch.randn(B, C, E)
W_q = nn.Linear(E, H * D, bias=False)
W_k = nn.Linear(E, H * D, bias=False)
W_v = nn.Linear(E, H * D, bias=False)

Q = W_q(x)
K = W_k(x)
V = W_v(x)
Q = Q.view(B, C, H, D).transpose(1, 2)
K = K.view(B, C, H, D).transpose(1, 2)
V = V.view(B, C, H, D).transpose(1, 2)

scores = torch.matmul(Q, K.transpose(-2, -1))
scores = scores / (D ** 0.5)
mask = torch.tril(torch.ones(C, C, device=scores.device))
scores = scores.masked_fill(mask == 0, float('-inf'))
weights = F.softmax(scores, dim=-1)
hidden = torch.matmul(weights, V)
hidden = hidden.transpose(1, 2).contiguous().view(B, C, H * D)
W_o = nn.Linear(H * D, E, bias=False)
output = W_o(hidden)
print('중간 구현 출력 shape:', output.shape)

중간 구현 출력 shape: torch.Size([2, 4, 16])


In [12]:
mha = MultiHeadAttention(embed_dim=512, n_head=8, head_dim=64)
x = torch.randn(2, 10, 512)
output = mha(x)
print('입력 shape:', x.shape)
print('출력 shape:', output.shape)

입력 shape: torch.Size([2, 10, 512])
출력 shape: torch.Size([2, 10, 512])


## `ch02/09_norm_gelu.py`

스크립트의 `__file__` 기반 경로 이동은 노트북에서는 필요하지 않으므로 제외하고, 실제 모델 구성 요소를 확인합니다.

In [13]:
class LayerNorm(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(embed_dim))
        self.beta = nn.Parameter(torch.zeros(embed_dim))
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * norm_x + self.beta

class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

class FFN(nn.Module):
    def __init__(self, x_dim, hidden_dim=None, dropout_rate=0.1):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = int(4 * x_dim)
        self.layers = nn.Sequential(
            nn.Linear(x_dim, hidden_dim),
            GELU(),
            nn.Linear(hidden_dim, x_dim),
            nn.Dropout(dropout_rate),
        )

    def forward(self, x):
        return self.layers(x)

class Block(nn.Module):
    def __init__(self, embed_dim, n_head, ff_dim=None, dropout_rate=0.1):
        super().__init__()
        head_dim = embed_dim // n_head
        self.norm1 = LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_head, head_dim, dropout_rate)
        self.norm2 = LayerNorm(embed_dim)
        self.ffn = FFN(embed_dim, ff_dim, dropout_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

## `ch02/10_gpt2.py`

In [14]:
class GPT(nn.Module):
    def __init__(
        self, vocab_size, max_context_len, embed_dim,
        n_head, n_layer, ff_dim, dropout_rate,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_context_len = max_context_len
        self.embed_dim = embed_dim
        self.n_head = n_head
        self.n_layer = n_layer
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate

        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_context_len, embed_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.blocks = nn.ModuleList([
            Block(embed_dim, n_head, ff_dim, dropout_rate)
            for _ in range(n_layer)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, vocab_size)
        self.embed.weight = self.unembed.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ids):
        B, C = ids.shape
        pos = torch.arange(0, C, dtype=torch.long, device=ids.device)
        x = self.dropout(self.embed(ids) + self.pos_embed(pos))
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.unembed(x)

    def save(self, file_path):
        checkpoint = {
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_context_len': self.max_context_len,
            'embed_dim': self.embed_dim,
            'n_head': self.n_head,
            'n_layer': self.n_layer,
            'ff_dim': self.ff_dim,
            'dropout_rate': self.dropout_rate,
        }
        torch.save(checkpoint, file_path)

    @classmethod
    def load_from(cls, file_path, device='cpu'):
        checkpoint = torch.load(file_path, map_location=device)
        model = cls(
            checkpoint['vocab_size'],
            checkpoint['max_context_len'],
            checkpoint['embed_dim'],
            checkpoint['n_head'],
            checkpoint['n_layer'],
            checkpoint['ff_dim'],
            checkpoint['dropout_rate'],
        )
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
        return model

vocab_size = 1000
max_context_len = 256
embed_dim = 384
n_head = 6
n_layer = 6
ff_dim = 4 * embed_dim
dropout_rate = 0.1

model = GPT(
    vocab_size, max_context_len, embed_dim,
    n_head, n_layer, ff_dim, dropout_rate,
)
dummy_input = torch.randint(0, vocab_size, (1, max_context_len))
logits = model(dummy_input)
print('출력 shape:', logits.shape)

출력 shape: torch.Size([1, 256, 1000])


## `ch02/graph.py`

In [15]:
x = np.linspace(-3, 3, 500)
relu = np.maximum(0, x)
gelu = 0.5 * x * (
    1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3))
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x, relu, 'b-', linewidth=2, label='ReLU')
ax.plot(x, gelu, 'r-', linewidth=2, label='GELU')
ax.set_xlim(-3, 3)
ax.set_ylim(-0.5, 3.0)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('f(x)', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.7)
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)
ax.legend(loc='upper left', fontsize=12)
plt.tight_layout()
plt.show()

## 실행 메모

모델 구조나 텐서 크기를 임의로 축소하지 않았습니다. 셀은 한 줄씩 분할하지 않고, 서로 함께 읽어야 하는 계산을 한 셀에 묶었습니다.